# tech-history 음성 생산 노트북 (Chatterbox Multilingual, 무료 T4 GPU)

**사용법 (순서 중요!)**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 저장
2. 아래 **첫 번째 코드 셀만** 실행(셀 왼쪽 ▶ 클릭) — 설치 (2~3분)
3. 메뉴 → 런타임 → **세션 다시 시작** (설치한 부품을 제대로 물리는 재시동 — 필수)
4. 메뉴 → 런타임 → **모두 실행** (첫 실행은 모델 다운로드로 5~10분 소요)
5. 마지막 셀이 끝나면 `voice_01.zip` 이 자동 다운로드됨 → `video/output/01_v2/audio/` 에 압축 풀기

(선택) 본인 목소리로 낭독시키려면: 왼쪽 폴더 아이콘 → `ref.wav`(5~10초 육성 녹음) 업로드 후 4번 실행 — 자동으로 그 목소리를 복제해 낭독합니다.

In [ ]:
EPISODE = "01"  # 생산할 편 번호
!pip install -q chatterbox-tts requests
!pip uninstall -y -q torchvision  # 미사용 부품 — 구버전 torch와 충돌하므로 제거
print("설치 완료 — 이제 메뉴에서 [런타임 → 세션 다시 시작] 후 [모두 실행] 하세요")

In [ ]:
import json, os, requests, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

url = f"https://raw.githubusercontent.com/nous-zero/tech-history/main/video/scripts/{EPISODE}.json"
script = requests.get(url).json()
print("대본:", script["title"], "/ 문단", len(script["segments"]))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)
model = ChatterboxMultilingualTTS.from_pretrained(device=device)

In [ ]:
ONLY = [1, 11, 14]  # 이 번호 세그먼트만 재생산 (빈 목록 [] 이면 전부 생산)

# --- 숫자 → 한글 발음 변환 (TTS 입력 전용 — 화면 자막은 원문 숫자 유지) ---
# Chatterbox가 아라비아 숫자를 한국어로 못 읽는 문제의 근본 수정.
import re
_SINO = "영일이삼사오육칠팔구"

def _sino(n):
    n = int(n)
    if n == 0:
        return "영"
    out = ""
    for val, name in ((10000, "만"), (1000, "천"), (100, "백"), (10, "십"), (1, "")):
        d, n = n // val, n % val
        if d:
            out += ("" if d == 1 and name else _SINO[d]) + name
    return out

_MONTH = {6: "유", 10: "시"}  # 6월=유월, 10월=시월 (특수 발음)

def normalize_numbers(t):
    t = re.sub(r"(\d+)월", lambda m: (_MONTH.get(int(m.group(1))) or _sino(m.group(1))) + "월", t)
    return re.sub(r"\d+", lambda m: _sino(m.group(0)), t)

REF = "ref.wav" if os.path.exists("ref.wav") else None  # 육성 복제용(선택)
out_dir = f"voice_{EPISODE}"
os.makedirs(out_dir, exist_ok=True)
for seg in script["segments"]:
    if ONLY and seg["id"] not in ONLY:
        continue
    text = normalize_numbers(seg["text"])
    kwargs = {"language_id": "ko"}
    if REF:
        kwargs["audio_prompt_path"] = REF
    # 비정상 길이 감시: 글자당 0.25초 + 여유 5초를 넘으면 환각(반복 생성)으로 보고 재시도
    limit = 0.25 * len(text) + 5
    for attempt in range(3):
        wav = model.generate(text, **kwargs)
        sec = wav.shape[-1] / model.sr
        if sec <= limit:
            break
        print(f"  seg{seg['id']:03d} {sec:.1f}초 — 비정상(기준 {limit:.0f}초), 재시도 {attempt + 1}/3")
    path = os.path.join(out_dir, f"seg{seg['id']:03d}.wav")
    torchaudio.save(path, wav, model.sr)
    print(f"seg{seg['id']:03d} 완료 ({sec:.1f}초)")
print("합성 완료:", "전체" if not ONLY else f"세그먼트 {ONLY}")

In [ ]:
import shutil
zip_path = shutil.make_archive(f"voice_{EPISODE}", "zip", out_dir)
from google.colab import files
files.download(zip_path)